# SEN-HARP Core v0.4 — lancement minimal des sept expériences

Ce notebook ne contient pas de logique économique. Il charge la configuration JSON, instancie le modèle, exécute les sept expériences et exporte les données brutes du collector.

Les résultats sont écrits dans `outputs/seven_experiments/<experiment_id>/`.


In [1]:
from pathlib import Path
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from dataclasses import asdict

import pandas as pd


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "senharp_core").is_dir()
            and (candidate / "experiments" / "seven_experiments.json").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "Projet introuvable. Lancez ce notebook depuis le dépôt SEN-HARP "
        "contenant senharp_core/ et experiments/seven_experiments.json."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "experiments" / "seven_experiments.json"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "seven_experiments"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Configuration:", CONFIG_PATH)
print("Outputs:", OUTPUT_ROOT)


Project root: C:\Users\pierr\Downloads\senharp-core-main\senharp-core-main
Configuration: C:\Users\pierr\Downloads\senharp-core-main\senharp-core-main\experiments\seven_experiments.json
Outputs: C:\Users\pierr\Downloads\senharp-core-main\senharp-core-main\outputs\seven_experiments


In [3]:
from senharp_core.entities import PolicyMode, Scenario
from senharp_core.model import Model
from senharp_core.parameters import Parameters

with CONFIG_PATH.open("r", encoding="utf-8") as stream:
    CONFIG = json.load(stream)

CONFIG["model_version"], len(CONFIG["experiments"])


('SEN-HARP Core v0.5 — household carbon pricing, factual credit calibration and GDP accounting',
 7)

## Fonction d'exécution

La fonction ci-dessous ne définit aucun comportement économique. Tous les mécanismes restent dans `senharp_core/`.


In [4]:
def run_one_experiment(experiment: dict, config: dict) -> dict[str, pd.DataFrame]:
    parameter_values = dict(config.get("shared_parameter_overrides", {}))
    parameter_values.update(experiment.get("parameter_overrides", {}))

    params = Parameters(**parameter_values)

    model = Model(
        params=params,
        scenario=Scenario[experiment["scenario"]],
        policy_mode=PolicyMode[experiment["policy_mode"]],
        public_service_spending_growth=config["run_control"].get(
            "public_service_spending_growth", 0.0
        ),
    )

    number_of_periods = int(config["run_control"]["number_of_periods"])
    for period in range(number_of_periods):
        model.run_period(period=period)

    tables = {
        "macro": pd.DataFrame(model.collector.macro_rows),
        "firms": pd.DataFrame(model.collector.firm_rows),
        "households": pd.DataFrame(model.collector.household_rows),
        "government": pd.DataFrame(model.collector.government_rows),
    }

    metadata = {
        "experiment_id": experiment["id"],
        "experiment_label": experiment["label"],
        "scenario": experiment["scenario"],
        "policy_mode": experiment["policy_mode"],
    }

    for table in tables.values():
        for key, value in metadata.items():
            table[key] = value

    return tables


## Exécution et export

Par défaut, le notebook remplace le dossier de sortie d'une expérience lorsqu'il existe déjà. Le code source du modèle n'est jamais modifié.


In [5]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_summary = []

for experiment in CONFIG["experiments"]:
    experiment_dir = OUTPUT_ROOT / experiment["id"]

    if experiment_dir.exists():
        shutil.rmtree(experiment_dir)
    experiment_dir.mkdir(parents=True)

    print(f'Running {experiment["id"]}: {experiment["label"]}')
    tables = run_one_experiment(experiment, CONFIG)

    for table_name, table in tables.items():
        table.to_csv(experiment_dir / f"{table_name}.csv", index=False)

    with (experiment_dir / "experiment_config.json").open(
        "w", encoding="utf-8"
    ) as stream:
        json.dump(experiment, stream, indent=2, ensure_ascii=False)

    parameter_values = dict(CONFIG.get("shared_parameter_overrides", {}))
    parameter_values.update(experiment.get("parameter_overrides", {}))
    resolved_parameters = asdict(Parameters(**parameter_values))
    with (experiment_dir / "resolved_parameters.json").open(
        "w", encoding="utf-8"
    ) as stream:
        json.dump(resolved_parameters, stream, indent=2, ensure_ascii=False)

    run_summary.append(
        {
            "experiment_id": experiment["id"],
            "experiment_label": experiment["label"],
            "macro_rows": len(tables["macro"]),
            "firm_rows": len(tables["firms"]),
            "household_rows": len(tables["households"]),
            "government_rows": len(tables["government"]),
        }
    )

run_summary_df = pd.DataFrame(run_summary)
run_summary_df.to_csv(OUTPUT_ROOT / "run_summary.csv", index=False)
run_summary_df


Running E0_baseline: Baseline


Running E1_carbon_tax_fixed: Carbon Tax — fixed


Running E2_carbon_tax_vote: Carbon Tax — endogenous vote


Running E3_green_deal_fixed: Green Deal — fixed


Running E4_green_deal_vote: Green Deal — endogenous vote


Running E5_post_growth_fixed: Post-Growth — fixed


Running E6_post_growth_vote: Post-Growth — endogenous vote


,experiment_id,experiment_label,macro_rows,firm_rows,household_rows,government_rows
0,E0_baseline,Baseline,26,3120,15600,26
1,E1_carbon_tax_fixed,Carbon Tax — fixed,26,3120,15600,26
2,E2_carbon_tax_vote,Carbon Tax — endogenous vote,26,3120,15600,26
3,E3_green_deal_fixed,Green Deal — fixed,26,3120,15600,26
4,E4_green_deal_vote,Green Deal — endogenous vote,26,3120,15600,26
5,E5_post_growth_fixed,Post-Growth — fixed,26,3120,15600,26
6,E6_post_growth_vote,Post-Growth — endogenous vote,26,3120,15600,26


## Informations de reproductibilité

Cette cellule crée un relevé correspondant à l'exécution courante. Elle ne remplace pas le fichier `ENVIRONMENT.txt` versionné à la racine du dépôt.


In [6]:
pytest_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "--tb=short",
        "--cache-clear",
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

environment_lines = [
    f"Generated at: {datetime.now().astimezone().isoformat()}",
    f"Project root: {PROJECT_ROOT}",
    f"Python executable: {sys.executable}",
    f"Python version: {platform.python_version()}",
    f"Platform: {platform.platform()}",
    "Pytest command: python -m pytest -q --tb=short --cache-clear",
    f"Pytest return code: {pytest_result.returncode}",
    "Pytest stdout:",
    pytest_result.stdout.strip(),
    "Pytest stderr:",
    pytest_result.stderr.strip(),
]

(OUTPUT_ROOT / "run_environment.txt").write_text(
    "\n".join(environment_lines) + "\n",
    encoding="utf-8",
)

print(pytest_result.stdout)
print(pytest_result.stderr)
print("Return code:", pytest_result.returncode)


..................................................................       [100%]
66 passed in 12.50s


Return code: 0
